# 15 · Process images as they arrive

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/15_comm_process_on_arrival.ipynb)

sparrow-comm hands every received file to a **sink**: any object with `seen(item)` and
`commit(item)`. Notebooks 11-14 used the built-in `Store` as the sink and ran detection
afterwards. Here a small sink wraps the store and runs MegaDetector the moment an image is
committed, whichever channel delivered it. This is the hook an application such as Sparrow
Studio implements to plug sparrow-comm into its own database and processing.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
# Setup: install sparrow-comm (the field-data receivers) if it is missing. It needs Python 3.12+.
# To try a build that is not on PyPI, set SPARROW_COMM_SPEC before starting Jupyter, e.g.
#   SPARROW_COMM_SPEC="/path/to/sparrow_comm-0.1.0-py3-none-any.whl[ftps]"
import os

if sys.version_info < (3, 12):
    raise RuntimeError("sparrow-comm needs Python 3.12 or newer, this kernel runs " + sys.version.split()[0])
for module, spec in [("sparrow_comm", os.environ.get("SPARROW_COMM_SPEC", "sparrow-comm[ftps]>=0.1.0")),
                     ("pyftpdlib", "pyftpdlib>=2.0"), ("OpenSSL", "pyopenssl>=24"), ("httpx", "httpx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import sparrow_comm
from sparrow_comm.store import Store

print("sparrow-comm", sparrow_comm.__version__)

In [ ]:
# A fresh workspace for this notebook: a SQLite database plus a folder of received files.
# Store implements all three sparrow-comm hooks (registry, config, sink). In an app such as
# Sparrow Studio, the app's own database implements them instead.
WORK = Path("comm_workspace") / "15_on_arrival"
shutil.rmtree(WORK, ignore_errors=True)
store = Store(WORK / "comm.db", WORK / "received")
PROJECT = 1

fetch("demo_data_det.zip")
IMAGES = sorted((DATA / "data/data_example/images").rglob("*.jpg"))[:4]
print(len(IMAGES), "demo images to send:", [p.name for p in IMAGES])

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

## A sink that detects on arrival

In [ ]:
from sparrow_comm import CommitResult, Item


class DetectOnArrival:
    """Store every item, then run MegaDetector on new images and keep the result."""

    def __init__(self, store, model="MDV6-yolov10-c", threshold=0.3):
        self.store, self.model, self.threshold = store, model, threshold
        self.results = {}  # item id -> DetectResult

    def seen(self, item: Item) -> bool:
        return self.store.seen(item)

    def commit(self, item: Item) -> CommitResult:
        res = self.store.commit(item)
        if res.created and item.kind == "image":
            row = next(r for r in self.store.items(item.project_id) if r["id"] == res.id)
            self.results[res.id] = spe.detect(self.store.data_dir / row["path"], model=self.model,
                                              threshold=self.threshold)[0]
            print(f"{item.channel:8} {item.original_name or item.name:28} ->",
                  len(self.results[res.id].detections), "detections")
        return res


sink = DetectOnArrival(store)

## Three channels, one sink

The device API and Reconyx endpoint share one app. FTPS staging is polled into the same sink.

In [ ]:
import base64
import json
import secrets

from fastapi import FastAPI
from fastapi.testclient import TestClient
from sparrow_comm.channels import ftps
from sparrow_comm.device_api import make_device_router
from sparrow_comm.reconyx import make_reconyx_router

app = FastAPI()
app.include_router(make_device_router(registry=store, sink=sink))
app.include_router(make_reconyx_router(config=store, sink=sink, jwt_secret=secrets.token_hex(32)))
api = TestClient(app)

# Device
key = store.issue_onboarding_key(PROJECT, "ranger@example.org")
api.post("/v1/onboarding", json={"unit_id": "unit-7"}, headers={"X-API-Key": key, "X-Email": "ranger@example.org"})
api.post("/v1/uploads/image", files={"file": (IMAGES[0].name, IMAGES[0].read_bytes(), "image/jpeg")},
         data={"metadata": json.dumps({"image_name": IMAGES[0].name})},
         headers={"X-API-Key": key, "X-Unit-ID": "unit-7"})

# Reconyx
token = store.issue_reconyx_key(PROJECT)
jwt = api.post("/v1/reconyx/login", json={"Id": f"proj_{PROJECT}", "Token": token}).json()["token"]
api.post("/v1/reconyx/images", headers={"api_key": jwt}, json={
    "Camera_serial_number": "HF2X-0042", "Image": base64.b64encode(IMAGES[1].read_bytes()).decode(),
    "Metadata": {"File_name": IMAGES[1].name}})

# FTPS: a camera dropped two files in the project's staging folder (notebook 12 shows the server)
store.set_ftps_password(PROJECT)
STAGING = WORK / "staging"
(STAGING / str(PROJECT)).mkdir(parents=True)
for i, img in enumerate(IMAGES[2:]):
    shutil.copy(img, STAGING / str(PROJECT) / f"TRAILCAM01_2026010{i + 1}063000.jpg")
ftps.poll_once(sink, store, STAGING, min_age_seconds=0)

# The same device image again: already seen, so no second detection run
api.post("/v1/uploads/image", files={"file": (IMAGES[0].name, IMAGES[0].read_bytes(), "image/jpeg")},
         data={"metadata": "{}"}, headers={"X-API-Key": key, "X-Unit-ID": "unit-7"})
print(len(sink.results), "images processed")

## Results

In [ ]:
import pandas as pd

rows = []
for r in store.items(PROJECT):
    res = sink.results[r["id"]]
    rows.append({"channel": r["channel"], "camera": r["camera_name"], "file": r["original_name"],
                 "detections": len(res.detections), "labels": sorted({d.label for d in res.detections})})
pd.DataFrame(rows)

Export everything that arrived as one MegaDetector JSON file, ready for Timelapse or other review tools.

In [ ]:
spe.export(list(sink.results.values()), format="megadet", output=WORK / "detections.json",
           model_id="MDV6-yolov10-c")
print((WORK / "detections.json").read_text()[:400])

In [ ]:
from IPython.display import Image, display

for png in spe.visualize(list(sink.results.values())[:2]):
    display(Image(data=png, width=480))

## Where to go from here

| Need | Do |
|---|---|
| Species names | Use `spe.pipeline(..., classifier=...)` in `commit` (notebook 03) |
| Your own database | Implement `seen`/`commit` (and the registry/config hooks) on your models |
| Long-running service | `sparrow-comm serve api`, `serve ftps`, `poll ftps`, `poll imap` |